# 08 — DNA Hybrid Model (Sequence CNN + Baseline Features)

This notebook trains a **hybrid DNA promoter classifier** that combines:

1. **Sequence-based representation**
   - One-hot encoded DNA sequences
   - 1D CNN to learn motif-level patterns

2. **Handcrafted baseline features**
   - GC content, CpG density, CpG O/E
   - Entropy, skew measures
   - k-mer frequencies (k=3)

The two representations are **fused via feature concatenation** and passed through a joint classification head.

This model is expected to outperform:
- Pure baseline ML models (Notebook 06)
- Pure sequence CNN (Notebook 07)

## Imports

In [1]:
from pathlib import Path
import json
import random
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

## Paths and Configuration

This notebook assumes:
- `notebooks/` is the current working directory
- All data artifacts were generated in previous notebooks

Configuration values are read from `configs/config.yaml`.

In [2]:
import yaml

ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
PROCESSED = DATA / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
MODELS = ROOT / "models" / "dna" / "hybrid"
CONFIGS = ROOT / "configs"

for p in [REPORTS, FIGURES, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

cfg_path = CONFIGS / "config.yaml"
assert cfg_path.exists(), f"Missing config: {cfg_path}"

with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

SEED = int(cfg["project"]["random_seed"])
L = int(cfg["dna"]["seq_length_bp"])
N_POS = int(cfg["dna"]["n_pos"])
N_NEG = int(cfg["dna"]["n_neg"])

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

(SEED, L, N_POS, N_NEG, str(device))

(42, 200, 2000, 2000, 'cpu')

## Load Sequence Dataset and Baseline Feature Dataset

We load:
- DNA sequence CSV from Notebook 03
- Baseline feature CSV from Notebook 05

Both files must have **matching row order and labels**.

In [3]:
seq_path = PROCESSED / f"dna_promoter_vs_nonpromoter_len{L}_pos{N_POS}_neg{N_NEG}.csv"
feat_path = PROCESSED / f"dna_features_baseline_len{L}_pos{N_POS}_neg{N_NEG}.csv"

assert seq_path.exists(), seq_path
assert feat_path.exists(), feat_path

df_seq = pd.read_csv(seq_path)
df_feat = pd.read_csv(feat_path)

df_seq.shape, df_feat.shape

((4000, 6), (4000, 100))

## Sanity Checks

We verify:
- Matching number of rows
- Balanced labels
- No missing values

In [4]:
assert df_seq.shape[0] == df_feat.shape[0]
assert (df_seq["label"].values == df_feat["label"].values).all()

df_seq.isna().sum(), df_feat.isna().sum(), df_seq["label"].value_counts()

(region_id    0
 chrom        0
 start        0
 end          0
 sequence     0
 label        0
 dtype: int64,
 region_id     0
 chrom         0
 start         0
 end           0
 label         0
              ..
 kmer_3_TGT    0
 kmer_3_TTA    0
 kmer_3_TTC    0
 kmer_3_TTG    0
 kmer_3_TTT    0
 Length: 100, dtype: int64,
 label
 1    2000
 0    2000
 Name: count, dtype: int64)

## One-Hot Encoding DNA Sequences

Each sequence is encoded into shape:
- `(L, 5)` for alphabet `{A, C, G, T, N}`

In [5]:
ALPHABET = {"A":0, "C":1, "G":2, "T":3, "N":4}

def one_hot_encode(seq, L):
    arr = np.zeros((L, 5), dtype=np.float32)
    for i, ch in enumerate(seq):
        arr[i, ALPHABET[ch]] = 1.0
    return arr

X_seq = np.stack(df_seq["sequence"].apply(lambda s: one_hot_encode(s, L)))
y = df_seq["label"].values.astype(np.float32)

X_seq.shape, y.mean()

((4000, 200, 5), np.float32(0.5))

## Prepare Baseline Feature Matrix

We:
- Drop metadata columns
- Standardize features using `StandardScaler`

In [6]:
meta_cols = ["region_id", "chrom", "start", "end", "label"]
X_feat = df_feat.drop(columns=meta_cols).values

scaler = StandardScaler()
X_feat = scaler.fit_transform(X_feat)

joblib.dump(scaler, MODELS / f"feature_scaler_len{L}.pkl")

X_feat.shape

(4000, 95)

## Train / Validation / Test Split

We use:
- 68% train
- 12% validation
- 20% test

Stratified by label.

In [7]:
X_seq_tr, X_seq_tmp, X_feat_tr, X_feat_tmp, y_tr, y_tmp = train_test_split(
    X_seq, X_feat, y, test_size=0.32, random_state=SEED, stratify=y
)

X_seq_val, X_seq_te, X_feat_val, X_feat_te, y_val, y_te = train_test_split(
    X_seq_tmp, X_feat_tmp, y_tmp, test_size=0.625, random_state=SEED, stratify=y_tmp
)

X_seq_tr.shape, X_seq_val.shape, X_seq_te.shape

((2720, 200, 5), (480, 200, 5), (800, 200, 5))

## PyTorch Dataset for Hybrid Model

In [8]:
class HybridDNADataset(Dataset):
    def __init__(self, X_seq, X_feat, y):
        self.X_seq = torch.tensor(X_seq)
        self.X_feat = torch.tensor(X_feat, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_feat[idx], self.y[idx]

## Hybrid CNN Architecture

- Sequence CNN branch → motif features
- Feature MLP branch → dense features
- Concatenation → joint classifier

In [9]:
class HybridCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()

        # Sequence branch
        self.conv = nn.Sequential(
            nn.Conv1d(5, 64, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1)
        )

        # Feature branch
        self.feat_fc = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU()
        )

        # Fusion head
        self.classifier = nn.Sequential(
            nn.Linear(128 + 128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x_seq, x_feat):
        x_seq = x_seq.permute(0, 2, 1)
        x_seq = self.conv(x_seq).squeeze(-1)
        x_feat = self.feat_fc(x_feat)
        x = torch.cat([x_seq, x_feat], dim=1)
        return self.classifier(x).squeeze(1)

## Training Setup

In [10]:
model = HybridCNN(X_feat.shape[1]).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_loader = DataLoader(HybridDNADataset(X_seq_tr, X_feat_tr, y_tr), batch_size=64, shuffle=True)
val_loader   = DataLoader(HybridDNADataset(X_seq_val, X_feat_val, y_val), batch_size=128)
test_loader  = DataLoader(HybridDNADataset(X_seq_te, X_feat_te, y_te), batch_size=128)

## Model Training

In [11]:
history = []

for epoch in range(1, 11):
    model.train()
    train_loss = 0.0

    for xs, xf, yb in train_loader:
        xs, xf, yb = xs.to(device), xf.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xs, xf)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for xs, xf, yb in val_loader:
            xs, xf = xs.to(device), xf.to(device)
            logits = model(xs, xf)
            probs.append(torch.sigmoid(logits).cpu().numpy())
            ys.append(yb.numpy())

    probs = np.concatenate(probs)
    ys = np.concatenate(ys)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss / len(train_loader),
        "val_roc_auc": roc_auc_score(ys, probs),
        "val_pr_auc": average_precision_score(ys, probs),
    })

pd.DataFrame(history)

,epoch,train_loss,val_roc_auc,val_pr_auc
0,1,0.568366,0.793108,0.815776
1,2,0.516674,0.801927,0.821198
2,3,0.498798,0.804549,0.818715
3,4,0.477563,0.807049,0.820568
4,5,0.465692,0.810017,0.822890
5,6,0.433376,0.812622,0.827658
6,7,0.400335,0.808038,0.822027
7,8,0.327214,0.818333,0.829614
8,9,0.230756,0.794878,0.811309
9,10,0.212107,0.792674,0.803420


## Final Test Evaluation and Saving Artifacts

In [12]:
model.eval()
probs, ys = [], []

with torch.no_grad():
    for xs, xf, yb in test_loader:
        xs, xf = xs.to(device), xf.to(device)
        logits = model(xs, xf)
        probs.append(torch.sigmoid(logits).cpu().numpy())
        ys.append(yb.numpy())

probs = np.concatenate(probs)
ys = np.concatenate(ys)
preds = (probs >= 0.5).astype(int)

results = {
    "roc_auc": roc_auc_score(ys, probs),
    "pr_auc": average_precision_score(ys, probs),
    "accuracy": accuracy_score(ys, preds),
    "precision": precision_score(ys, preds),
    "recall": recall_score(ys, preds),
    "f1": f1_score(ys, preds),
    "n_test": len(ys),
}

results

{'roc_auc': 0.8316875000000001,
 'pr_auc': 0.8466018777391506,
 'accuracy': 0.6975,
 'precision': 0.6484962406015038,
 'recall': 0.8625,
 'f1': 0.740343347639485,
 'n_test': 800}

## Save Model and Summary

In [13]:
torch.save(model.state_dict(), MODELS / f"hybrid_len{L}.pt")

with open(REPORTS / f"dna_hybrid_summary_len{L}.json", "w") as f:
    json.dump(results, f, indent=2)

results

{'roc_auc': 0.8316875000000001,
 'pr_auc': 0.8466018777391506,
 'accuracy': 0.6975,
 'precision': 0.6484962406015038,
 'recall': 0.8625,
 'f1': 0.740343347639485,
 'n_test': 800}